# 00 Data Prep

This notebook imports and loads data from different sources and cosolidates it.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.font_manager as fm
import yfinance as yf
from scipy.stats import jarque_bera

## 0.1 Raw data acquisition

- **(a)** target-domain commodity series (the dependent variable's source)
- **(b)** source-domain analog of the same commodity
- **(c)** shared-core global/macro-financial predictors — must be *one* series each, used identically in both domains
- **(d)** target-only local predictors

### 0.1(a) Target-domain commodity series

DCE (Dalian Commodity Exchange)

In [20]:
RAW_DIR = Path("../2_data/2.1_raw/DCE")
PROCESSED_DIR = Path("../2_data/2.2_processed/future_contract_data")

RENAME = {
    "Products": "commodity_name", "Contract": "contract", "Trade Date": "trade_date",
    "Open": "open", "High": "high", "Low": "low", "Close": "close",
    "Prev Settle": "prev_settle", "Settle": "settle", "Chg": "chg",
    "Change1": "settle_chg", "Volume": "volume", "OI": "oi",
    "OI Chg": "oi_chg", "Turnover": "turnover",
}
NUMERIC_COLS = [
    "open", "high", "low", "close", "prev_settle", "settle",
    "chg", "settle_chg", "volume", "oi", "oi_chg", "turnover",
]

In [21]:
def load_product_year(path: Path, commodity_code: str) -> pd.DataFrame:
    """Read one <code>_ftr.xlsx file into a cleaned long-format frame; empty files -> empty frame."""
    df = pd.read_excel(path, sheet_name="HistoryDayQuotes", dtype=str)
    if df.empty:
        return df

    df = df.rename(columns=RENAME)
    for col in NUMERIC_COLS:
        df[col] = pd.to_numeric(df[col].str.replace(",", "", regex=False), errors="coerce")

    df["trade_date"] = pd.to_datetime(df["trade_date"], format="%Y%m%d")
    df["commodity"] = commodity_code  # stable identifier taken from the filename, not the free-text name

    # Contract code is <prefix letters><YYMM><optional suffix>, e.g. "c2103" or "l2602F" -> 2026-02.
    ym = df["contract"].str.extract(r"(\d{4})")[0]
    df["contract_year"] = 2000 + ym.str[:2].astype(int)
    df["contract_month"] = ym.str[2:].astype(int)

    return df

In [22]:
frames = []
for year_dir in sorted(p for p in RAW_DIR.iterdir() if p.is_dir()):
    for file in sorted(year_dir.glob("*_ftr.xlsx")):
        code = file.stem[: -len("_ftr")]
        frame = load_product_year(file, code)
        if not frame.empty:
            frames.append(frame)

panel = pd.concat(frames, ignore_index=True)

# Safety net: a contract can straddle two calendar-year files (e.g. c2103 trades through both the
# 2020 and 2021 raw files), so guard against any accidental exact-duplicate rows on the natural key.
panel = (
    panel.drop_duplicates(subset=["commodity", "contract", "trade_date"])
    .sort_values(["commodity", "contract", "trade_date"])
    .reset_index(drop=True)
)

cols = [
    "commodity", "commodity_name", "contract", "contract_year", "contract_month", "trade_date",
    "open", "high", "low", "close", "prev_settle", "settle", "chg", "settle_chg",
    "volume", "oi", "oi_chg", "turnover",
]
panel = panel[cols]
panel.shape

c:\Users\Leo\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\Leo\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\Leo\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\Leo\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\Leo\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no defaul

(732137, 18)

In [23]:
# Sanity checks before persisting
print(f"{len(panel):,} rows | {panel['commodity'].nunique()} commodities | "
      f"{panel['trade_date'].min().date()} to {panel['trade_date'].max().date()}")

coverage = panel.groupby("commodity")["trade_date"].agg(n_rows="count", first="min", last="max")
coverage.sort_values("n_rows")

732,137 rows | 26 commodities | 2006-01-04 to 2026-09-02


,n_rows,first,last
commodity,,,
l-F,1176,2025-10-29,2026-09-02
pp-F,1176,2025-10-29,2026-09-02
v-F,1176,2025-10-29,2026-09-02
lg,2363,2024-11-18,2026-09-02
bz,2709,2025-07-08,2026-09-02
lh,7946,2021-01-08,2026-09-02
cs,17051,2014-12-19,2026-09-02
pg,18169,2020-03-30,2026-09-02
eb,19760,2019-09-26,2026-09-02


In [24]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

panel.to_csv(
    PROCESSED_DIR / "dce_futures_consolidated.csv",
    index=False,
)

print(f"Saved consolidated panel to {PROCESSED_DIR.resolve()}")

Saved consolidated panel to C:\Users\Leo\PycharmProjects\Thesis Project\2_data\2.2_processed\future_contract_data


In [32]:
INPUT_FILE = Path("../2_data/2.2_processed/future_contract_data/dce_futures_consolidated.csv")
OUTPUT_DIR = Path("../2_data/2.2_processed/target_series")

ROLL_DAYS = 5
INITIAL_INDEX = 100.0


def convert_to_template(source: pd.DataFrame) -> pd.DataFrame:
    """
    Convert the DCE source dataframe to:

        date
        commodity
        contract
        expiry_date
        settle
        volume
        open_interest

    The source does not contain an explicit expiry date. For completed
    contracts, the last available trading date is used as an expiry proxy.
    Contracts still trading at the dataset cutoff use delivery month-end.
    """
    required_columns = {
        "trade_date",
        "commodity",
        "contract",
        "contract_year",
        "contract_month",
        "settle",
        "volume",
        "oi",
    }

    missing = required_columns - set(source.columns)
    if missing:
        raise ValueError(
            f"Source is missing required columns: {sorted(missing)}"
        )

    template = source[
        [
            "trade_date",
            "commodity",
            "contract",
            "contract_year",
            "contract_month",
            "settle",
            "volume",
            "oi",
        ]
    ].rename(
        columns={
            "trade_date": "date",
            "oi": "open_interest",
        }
    )

    template["date"] = pd.to_datetime(
        template["date"],
        errors="raise",
    )
    template["settle"] = pd.to_numeric(
        template["settle"],
        errors="raise",
    )

    duplicate_key = ["date", "commodity", "contract"]

    if template.duplicated(duplicate_key).any():
        raise ValueError(
            "Duplicate date/commodity/contract observations found."
        )

    if template["settle"].isna().any():
        raise ValueError("Missing settlement prices found.")

    if template["settle"].le(0).any():
        raise ValueError("Settlement prices must be positive.")

    dataset_cutoff = template["date"].max()

    # Last observed trading date for each contract
    last_observed_date = (
        template.groupby(
            ["commodity", "contract"],
            observed=True,
        )["date"]
        .transform("max")
    )

    # Month-end of the contract delivery month
    delivery_month_start = pd.to_datetime(
        {
            "year": template["contract_year"],
            "month": template["contract_month"],
            "day": 1,
        },
        errors="raise",
    )

    delivery_month_end = (
        delivery_month_start + pd.offsets.MonthEnd(0)
    )

    template["expiry_date"] = last_observed_date

    # A contract whose last observation equals the dataset cutoff may
    # still be active. Do not treat the cutoff as its expiry date.
    live_at_cutoff = last_observed_date.eq(dataset_cutoff)

    template.loc[live_at_cutoff, "expiry_date"] = (
        delivery_month_end.loc[live_at_cutoff]
    )

    return template[
        [
            "date",
            "commodity",
            "contract",
            "expiry_date",
            "settle",
            "volume",
            "open_interest",
        ]
    ].copy()


def make_futures_series(
    df: pd.DataFrame,
    commodity: str,
    roll_days: int = 5,
    initial_index: float = 100.0,
) -> pd.DataFrame:
    """
    Construct a nearest-eligible-contract futures return index.

    The active contract is the contract with the nearest expiry date
    that has more than `roll_days` calendar days remaining.

    On a roll date, the return is calculated using the new contract's
    current and previous settlement prices. This prevents the price
    difference between two contracts from being counted as a return.
    """
    required_columns = {
        "date",
        "commodity",
        "contract",
        "expiry_date",
        "settle",
    }

    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(
            f"Template is missing required columns: {sorted(missing)}"
        )

    commodity_data = df.loc[
        df["commodity"].eq(commodity)
    ].copy()

    if commodity_data.empty:
        raise ValueError(
            f"No observations found for commodity {commodity!r}."
        )

    commodity_data["date"] = pd.to_datetime(
        commodity_data["date"]
    )
    commodity_data["expiry_date"] = pd.to_datetime(
        commodity_data["expiry_date"]
    )
    commodity_data["settle"] = pd.to_numeric(
        commodity_data["settle"],
        errors="raise",
    )

    if commodity_data.duplicated(["date", "contract"]).any():
        raise ValueError(
            f"Duplicate date/contract observations for {commodity}."
        )

    commodity_data["days_to_expiry"] = (
        commodity_data["expiry_date"]
        - commodity_data["date"]
    ).dt.days

    # Exclude contracts inside the roll window.
    eligible = commodity_data.loc[
        commodity_data["days_to_expiry"] > roll_days
    ].copy()

    # Select the nearest eligible contract on each date.
    active = (
        eligible
        .sort_values(
            ["date", "expiry_date", "contract"]
        )
        .groupby("date", as_index=False)
        .first()
        [
            [
                "date",
                "contract",
                "expiry_date",
                "settle",
            ]
        ]
        .sort_values("date")
    )

    # Create a date-by-contract price matrix.
    price_matrix = commodity_data.pivot(
        index="date",
        columns="contract",
        values="settle",
    )

    previous_price_matrix = price_matrix.shift(1)

    active = active.set_index("date")

    # Obtain the previous day's settlement for whichever contract is
    # active today.
    active["previous_same_contract_price"] = [
        previous_price_matrix.at[date, contract]
        if (
            date in previous_price_matrix.index
            and contract in previous_price_matrix.columns
        )
        else np.nan
        for date, contract in active["contract"].items()
    ]

    active["return"] = (
        active["settle"]
        / active["previous_same_contract_price"]
        - 1
    )

    active["roll"] = active["contract"].ne(
        active["contract"].shift()
    )

    # The first return is unavailable and is treated as zero so that
    # the index begins at INITIAL_INDEX.
    active["return_index"] = (
        1 + active["return"].fillna(0)
    ).cumprod() * initial_index

    return active.reset_index()


def main() -> None:
    source = pd.read_csv(
        INPUT_FILE,
        low_memory=False,
    )

    template = convert_to_template(source)

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    commodities = sorted(
        template["commodity"].dropna().unique()
    )

    for commodity in commodities:
        series = make_futures_series(
            df=template,
            commodity=commodity,
            roll_days=ROLL_DAYS,
            initial_index=INITIAL_INDEX,
        )

        output = series[
            ["date", "return_index"]
        ].rename(
            columns={
                "date": "data",
                "return_index": "value",
            }
        )

        output["data"] = output["data"].dt.strftime(
            "%Y-%m-%d"
        )

        output_file = (
            OUTPUT_DIR
            / f"{commodity}_futures_series.csv"
        )

        output.to_csv(
            output_file,
            index=False,
            float_format="%.10f",
        )

        print(
            f"Created {output_file} "
            f"with {len(output):,} observations"
        )

    print(f"Finished: {len(commodities)} series created.")


if __name__ == "__main__":
    main()

Created ..\2_data\2.2_processed\target_series\a_futures_series.csv with 5,023 observations
Created ..\2_data\2.2_processed\target_series\b_futures_series.csv with 5,023 observations
Created ..\2_data\2.2_processed\target_series\bb_futures_series.csv with 3,099 observations
Created ..\2_data\2.2_processed\target_series\bz_futures_series.csv with 283 observations
Created ..\2_data\2.2_processed\target_series\c_futures_series.csv with 5,023 observations
Created ..\2_data\2.2_processed\target_series\cs_futures_series.csv with 2,845 observations
Created ..\2_data\2.2_processed\target_series\eb_futures_series.csv with 1,681 observations
Created ..\2_data\2.2_processed\target_series\eg_futures_series.csv with 1,876 observations
Created ..\2_data\2.2_processed\target_series\fb_futures_series.csv with 3,099 observations
Created ..\2_data\2.2_processed\target_series\i_futures_series.csv with 3,134 observations
Created ..\2_data\2.2_processed\target_series\j_futures_series.csv with 3,740 observat

Zhengong Commodity Exchange

Multi Commodity Exchange (MCX) India

### 0.1(b) Source-domain commodity series

In [2]:
TICKERS = {
    "GC=F": "gold",
    "SI=F": "silver",
    "HG=F": "copper",
    "CL=F": "crude_oil_wti",
    "BZ=F": "crude_oil_brent",
    "NG=F": "natural_gas",
    "ZC=F": "corn",
    "ZO=F": "oats",
    "KE=F": "wheat",
    "ZR=F": "rough_rice",
    "ZS=F": "soybeans",
    "GF=F": "feeder_cattle",
    "HE=F": "lean_hogs",
    "LE=F": "live_cattle",
    "CC=F": "cocoa",
    "KC=F": "coffee",
    "CT=F": "cotton",
    "LBS=F": "lumber",
    "OJ=F": "orange_juice",
    "SB=F": "sugar",
}

OUT_DIR = Path("../2_data/2.1_raw/Yahoo Finance")
OUT_DIR.mkdir(parents=True, exist_ok=True)

for ticker, name in TICKERS.items():
    out_path = OUT_DIR / f"{ticker}_{name}_daily.csv"

    try:
        df = yf.download(
            ticker,
            period="max",
            interval="1d",
            auto_adjust=False,
            progress=False,
        )

        if df.empty:
            print(f"{ticker}: no data returned")
            continue

        df.index.name = "date"
        df.to_csv(out_path)

        print(
            f"{ticker}: {len(df):,} rows | "
            f"{df.index.min().date()} to {df.index.max().date()} | "
            f"Saved to {out_path.resolve()}"
        )

    except Exception as error:
        print(f"{ticker}: download failed — {error}")

GC=F: 6,529 rows | 2000-08-30 to 2026-09-08 | Saved to C:\Users\Leo\PycharmProjects\Thesis Project\2_data\2.1_raw\Yahoo Finance\GC=F_gold_daily.csv
SI=F: 6,531 rows | 2000-08-30 to 2026-09-08 | Saved to C:\Users\Leo\PycharmProjects\Thesis Project\2_data\2.1_raw\Yahoo Finance\SI=F_silver_daily.csv
HG=F: 6,534 rows | 2000-08-30 to 2026-09-08 | Saved to C:\Users\Leo\PycharmProjects\Thesis Project\2_data\2.1_raw\Yahoo Finance\HG=F_copper_daily.csv
CL=F: 6,538 rows | 2000-08-23 to 2026-09-08 | Saved to C:\Users\Leo\PycharmProjects\Thesis Project\2_data\2.1_raw\Yahoo Finance\CL=F_crude_oil_wti_daily.csv
BZ=F: 4,756 rows | 2007-07-30 to 2026-09-08 | Saved to C:\Users\Leo\PycharmProjects\Thesis Project\2_data\2.1_raw\Yahoo Finance\BZ=F_crude_oil_brent_daily.csv
NG=F: 6,535 rows | 2000-08-30 to 2026-09-08 | Saved to C:\Users\Leo\PycharmProjects\Thesis Project\2_data\2.1_raw\Yahoo Finance\NG=F_natural_gas_daily.csv
ZC=F: 6,541 rows | 2000-07-17 to 2026-09-08 | Saved to C:\Users\Leo\PycharmProjec

In [3]:
RAW_DIR = Path("../2_data/2.1_raw/Yahoo Finance")

FILES = {
    "Gold": "GC=F_gold_daily.csv",
    "Silver": "SI=F_silver_daily.csv",
    "Copper": "HG=F_copper_daily.csv",
    "WTI Crude Oil": "CL=F_crude_oil_wti_daily.csv",
    "Brent Crude Oil": "BZ=F_crude_oil_brent_daily.csv",
    "Natural Gas": "NG=F_natural_gas_daily.csv",
    "Corn": "ZC=F_corn_daily.csv",
    "Oats": "ZO=F_oats_daily.csv",
    "Wheat": "KE=F_wheat_daily.csv",
    "Rough Rice": "ZR=F_rough_rice_daily.csv",
    "Soybeans": "ZS=F_soybeans_daily.csv",
    "Feeder Cattle": "GF=F_feeder_cattle_daily.csv",
    "Lean Hogs": "HE=F_lean_hogs_daily.csv",
    "Live Cattle": "LE=F_live_cattle_daily.csv",
    "Cocoa": "CC=F_cocoa_daily.csv",
    "Coffee": "KC=F_coffee_daily.csv",
    "Cotton": "CT=F_cotton_daily.csv",
    "Lumber": "LBS=F_lumber_daily.csv",
    "Orange Juice": "OJ=F_orange_juice_daily.csv",
    "Sugar": "SB=F_sugar_daily.csv",
}

series = {}

for commodity, filename in FILES.items():
    path = RAW_DIR / filename

    df = pd.read_csv(
        path,
        header=[0, 1],
        index_col=0,
        parse_dates=True,
    )

    adjusted_close = df["Adj Close"].iloc[:, 0]
    adjusted_close.name = commodity
    series[commodity] = adjusted_close

source_series_level = pd.DataFrame(series)
source_series_level.index.name = "date"
source_series_level = source_series_level.sort_index()


print(source_series_level.info())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 6804 entries, 1999-09-14 to 2026-09-08
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Gold             6529 non-null   float64
 1   Silver           6531 non-null   float64
 2   Copper           6534 non-null   float64
 3   WTI Crude Oil    6538 non-null   float64
 4   Brent Crude Oil  4756 non-null   float64
 5   Natural Gas      6535 non-null   float64
 6   Corn             6541 non-null   float64
 7   Oats             6620 non-null   float64
 8   Wheat            6530 non-null   float64
 9   Rough Rice       6762 non-null   float64
 10  Soybeans         6533 non-null   float64
 11  Feeder Cattle    6365 non-null   float64
 12  Lean Hogs        6463 non-null   float64
 13  Live Cattle      6395 non-null   float64
 14  Cocoa            6691 non-null   float64
 15  Coffee           6689 non-null   float64
 16  Cotton           6691 non-null   float64
 

In [4]:
source_series = np.log(source_series_level).diff()

c:\Users\Leo\anaconda3\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)


In [8]:
summary_stats = source_series.describe()

jarque_bera_stat, jarque_bera_pvalue = jarque_bera(source_series, axis=0, nan_policy="omit")

additional_stats = pd.DataFrame({
    'skewness': source_series.skew(),
    'kurtosis': source_series.kurt() + 3,
    'variance': source_series.var(),
    'median': source_series.median(),
    'mode':source_series.mode().iloc[0],
    'sum': source_series.sum(),
    'sum of squared deviations': ((source_series - source_series.mean())**2).sum(),
    'jarque-bera': jarque_bera_stat,
    'jarque-bera probability value': jarque_bera_pvalue,
})

additional_stats = additional_stats.T

summary_stats = pd.concat([summary_stats, additional_stats], axis=0)
summary_stats.columns.name = None
summary_stats

,Gold,Silver,Copper,WTI Crude Oil,Brent Crude Oil,Natural Gas,Corn,Oats,Wheat,Rough Rice,Soybeans,Feeder Cattle,Lean Hogs,Live Cattle,Cocoa,Coffee,Cotton,Lumber,Orange Juice,Sugar
count,6501.000000,6505.000000,6511.000000,6513.000000,4716.000000,6513.000000,6508.000000,6.551000e+03,6.514000e+03,6733.000000,6517.000000,6338.000000,6443.000000,6377.000000,6661.000000,6658.000000,6661.000000,5711.000000,6240.000000,6624.000000
mean,0.000404,0.000375,0.000273,0.000247,0.000007,-0.000061,0.000153,-4.319553e-06,1.517644e-04,0.000150,0.000156,0.000208,0.000050,0.000152,0.000297,0.000129,0.000074,0.000072,0.000107,0.000180
std,0.011263,0.021119,0.017182,0.026273,0.024308,0.038433,0.017836,2.710839e-02,1.873260e-02,0.017200,0.015728,0.010334,0.023398,0.011374,0.022199,0.021716,0.018630,0.026031,0.023433,0.020699
min,-0.120657,-0.376103,-0.251712,-0.282206,-0.279761,-0.643974,-0.268620,-6.127906e-01,-8.994824e-02,-0.299703,-0.234109,-0.086114,-0.271578,-0.156477,-0.260570,-0.128467,-0.272925,-0.407641,-0.133244,-0.180382
25%,-0.004867,-0.008607,-0.008438,-0.012748,-0.010428,-0.020205,-0.009289,-1.101296e-02,-1.138122e-02,-0.008144,-0.007649,-0.004294,-0.007889,-0.004704,-0.011011,-0.012461,-0.009453,-0.013520,-0.011539,-0.011327
50%,0.000486,0.001018,0.000241,0.001113,0.000743,-0.000310,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000645,0.000309,0.000323,0.000333,0.000483,0.000000,0.000000,-0.000339,0.000428,0.000000
75%,0.006362,0.010496,0.009310,0.013612,0.011720,0.019289,0.009589,1.159117e-02,1.085517e-02,0.008360,0.008518,0.004889,0.008247,0.005710,0.011922,0.012021,0.009551,0.013054,0.012297,0.011845
max,0.086432,0.131250,0.124437,0.319634,0.190774,0.381727,0.127571,4.957877e-01,8.097700e-02,0.162479,0.203209,0.112786,0.236340,0.106347,0.128047,0.166313,0.136218,0.267733,0.197781,0.235470
skewness,-0.506161,-1.493587,-0.608300,-0.100483,-0.788039,-0.315739,-0.874597,-1.353913e+00,1.026493e-01,-1.794298,-0.915124,0.190425,-1.097437,-1.047138,-0.551367,0.220032,-0.459640,-0.205416,-0.026880,-0.017884
kurtosis,9.627278,24.069252,13.937357,18.416649,14.863243,21.375429,17.175311,1.011226e+02,4.539702e+00,39.640908,19.713715,14.569012,30.979349,16.625020,10.631623,5.709080,13.105446,23.709001,6.722819,8.896288


In [9]:
PROCESSED_DIR = Path("../2_data/2.2_processed")
OUTPUT_PATH = PROCESSED_DIR / "source_series.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
source_series.to_csv(OUTPUT_PATH, index=True)

print(f"{len(source_series):,} rows saved to {OUTPUT_PATH.resolve()}")

6,804 rows saved to C:\Users\Leo\PycharmProjects\Thesis Project\2_data\2.2_processed\source_series.csv


### 0.1(c) Shared-core global/macro-financial predictors

In [ ]:
df_source_features = pd.DataFrame(
    index=pd.date_range(
        start="1954-01-04",
        end=pd.Timestamp.today().normalize(),
        freq="D",
        name="date"
    )
)

df_source_features

Global commodity benchmark index: S&P GSCI (Daily, from 31.12.1969)

In [ ]:
data_folder = Path("../2_data/2.1_raw/Capital IQ")
files = sorted(data_folder.glob("*.csv"))

frames = []

for file in files:
    file_df = pd.read_csv(file)

    # Remove accidental whitespace from column names
    file_df.columns = file_df.columns.str.strip()

    # Convert Date strings such as 12/31/1969
    file_df["Date"] = pd.to_datetime(
        file_df["Date"],
        format="%m/%d/%Y",
        errors="coerce"
    )

    frames.append(file_df[["Date", "Close"]])

if not frames:
    raise FileNotFoundError(
        f"No CSV files found in: {data_folder.resolve()}"
    )

gsci = pd.concat(frames, ignore_index=True)

# Clean and index the combined series
gsci = (
    gsci
    .dropna(subset=["Date"])
    .drop_duplicates(subset=["Date"], keep="last")
    .set_index("Date")
    .sort_index()
)

# Ensure the target DataFrame also has a DatetimeIndex
df_source_features.index = pd.to_datetime(df_source_features.index)
df_source_features.index.name = "Date"

# Assignment aligns the values by date
df_source_features["GSCI"] = gsci["Close"]

df_source_features

Federal Funds Effective Rate: DFF (Daily, from 01.07.1954)

In [ ]:
path = Path("../2_data/2.1_raw/FRED/DFF.xlsx")

dff = pd.read_excel(
    path,
    sheet_name="Daily, 7-Day",
    usecols=["observation_date", "DFF"]
)

dff["observation_date"] = pd.to_datetime(dff["observation_date"])

dff = (
    dff
    .dropna(subset=["observation_date"])
    .drop_duplicates(subset=["observation_date"], keep="last")
    .set_index("observation_date")
    .sort_index()
)

# Make sure the existing index is also datetime
df_source_features.index = pd.to_datetime(df_source_features.index)
df_source_features.index.name = "Date"

# Pandas matches the rows by date
df_source_features["DFF"] = dff["DFF"]

df_source_features

3-Month Treasury Bill Secondary Market Rate: DTB3 (Daily, from 04.01.1954)

In [ ]:
path = Path("../2_data/2.1_raw/FRED/DTB3.xlsx")

dtb3 = pd.read_excel(
    path,
    sheet_name="Daily",
    usecols=["observation_date", "DTB3"]
)

dtb3["observation_date"] = pd.to_datetime(dtb3["observation_date"])

dtb3 = (
    dtb3
    .dropna(subset=["observation_date"])
    .drop_duplicates(subset=["observation_date"], keep="last")
    .set_index("observation_date")
    .sort_index()
)

# Make sure the existing index is also datetime
df_source_features.index = pd.to_datetime(df_source_features.index)
df_source_features.index.name = "Date"

# Pandas matches the rows by date
df_source_features["DTB3"] = dtb3["DTB3"]

df_source_features

Nominal Broad U.S. Dollar Index: DTWEXBGS (Daily, from 02.01.2006)

In [ ]:
path = Path("../2_data/2.1_raw/FRED/DTWEXBGS.xlsx")

dtwexbgs = pd.read_excel(
    path,
    sheet_name="Daily",
    usecols=["observation_date", "DTWEXBGS"]
)

dtwexbgs["observation_date"] = pd.to_datetime(dtwexbgs["observation_date"])

dtwexbgs = (
    dtwexbgs
    .dropna(subset=["observation_date"])
    .drop_duplicates(subset=["observation_date"], keep="last")
    .set_index("observation_date")
    .sort_index()
)

# Make sure the existing index is also datetime
df_source_features.index = pd.to_datetime(df_source_features.index)
df_source_features.index.name = "Date"

# Pandas matches the rows by date
df_source_features["DTWEXBGS"] = dtwexbgs["DTWEXBGS"] 

df_source_features

CBOE Volatility Index: VIXCLS (Daily, from 02.01.1990)

In [ ]:
path = Path("../2_data/2.1_raw/FRED/VIXCLS.xlsx") 

vixcls = pd.read_excel(
    path,
    sheet_name="Daily, Close",
    usecols=["observation_date", "VIXCLS"]
)

vixcls["observation_date"] = pd.to_datetime(vixcls["observation_date"])

vixcls = (
    vixcls
    .dropna(subset=["observation_date"])
    .drop_duplicates(subset=["observation_date"], keep="last")
    .set_index("observation_date")
    .sort_index()
)

# Make sure the existing index is also datetime
df_source_features.index = pd.to_datetime(df_source_features.index)
df_source_features.index.name = "Date"

# Pandas matches the rows by date
df_source_features["VIXCLS"] = vixcls["VIXCLS"]

df_source_features

In [ ]:
df_source_features["VIXCLS"].notna().sum()

In [ ]:
output_path = Path("../2_data/2.2_processed/source_features.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)

df_source_features.to_csv(output_path, index=True)

print(f"Saved to: {output_path.resolve()}")

### 0.1(d) Target-only local predictors

China
- Nominal effective exchange rate (Yuan vs. broad basket) from BIS
- Local inflation (CPI growth, monthly!) from FRED
- Local policy rate from BIS
- Local stock market index (SSE Composite Index) from Yahoo Finance

In [10]:
df_target_features_china = pd.DataFrame(
    index=pd.date_range(
        start="1954-01-04",
        end=pd.Timestamp.today().normalize(),
        freq="D",
        name="date"
    )
)

In [11]:
urls = ["https://stats.bis.org/api/v2/data/dataflow/BIS/WS_EER/1.0/D.N.B.CN?format=csv"]

df = pd.concat([pd.read_csv(url) for url in urls])

output_path = Path("../2_data/2.1_raw/BIS/D_N_B_CN.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=True)

print(f"Saved to: {output_path.resolve()}")

Saved to: C:\Users\Leo\PycharmProjects\Thesis Project\2_data\2.1_raw\BIS\D_N_B_CN.csv


In [12]:
df_target_features_china.index = pd.to_datetime(df_target_features_china.index)

df_target_features_china["FX rate"] = (
    df.assign(TIME_PERIOD=pd.to_datetime(df["TIME_PERIOD"]))
      .set_index("TIME_PERIOD")["OBS_VALUE"]
      .reindex(df_target_features_china.index)
)
df_target_features_china

,FX rate
date,
1954-01-04,NaN
1954-01-05,NaN
1954-01-06,NaN
1954-01-07,NaN
1954-01-08,NaN
...,...
2026-09-07,112.53
2026-09-08,112.44
2026-09-09,NaN


In [13]:
path = Path("../2_data/2.1_raw/FRED/CPALTT01CNM657N.csv")

cpi_growth = pd.read_csv(
    path,
    usecols=["observation_date", "CPALTT01CNM657N"]
)

cpi_growth["observation_date"] = pd.to_datetime(cpi_growth["observation_date"])

cpi_growth = (
    cpi_growth
    .dropna(subset=["observation_date"])
    .drop_duplicates(subset=["observation_date"], keep="last")
    .set_index("observation_date")
    .sort_index()
)

# Make sure the existing index is also datetime
df_target_features_china.index = pd.to_datetime(df_target_features_china.index)
df_target_features_china.index.name = "Date"

# Pandas matches the rows by date
df_target_features_china["CPI growth"] = cpi_growth["CPALTT01CNM657N"]

df_target_features_china

,FX rate,CPI growth
Date,,
1954-01-04,NaN,NaN
1954-01-05,NaN,NaN
1954-01-06,NaN,NaN
1954-01-07,NaN,NaN
1954-01-08,NaN,NaN
...,...,...
2026-09-07,112.53,NaN
2026-09-08,112.44,NaN
2026-09-09,NaN,NaN


In [14]:
urls = ["https://stats.bis.org/api/v2/data/dataflow/BIS/WS_CBPOL/1.0/D.CN?format=csv"]

df = pd.concat([pd.read_csv(url) for url in urls])

output_path = Path("../2_data/2.1_raw/BIS/D_CN.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=True)

print(f"Saved to: {output_path.resolve()}")

Saved to: C:\Users\Leo\PycharmProjects\Thesis Project\2_data\2.1_raw\BIS\D_CN.csv


In [15]:
df_target_features_china.index = pd.to_datetime(df_target_features_china.index)

df_target_features_china["Local policy rate"] = (
    df.assign(TIME_PERIOD=pd.to_datetime(df["TIME_PERIOD"]))
      .set_index("TIME_PERIOD")["OBS_VALUE"]
      .reindex(df_target_features_china.index)
)
df_target_features_china

,FX rate,CPI growth,Local policy rate
Date,,,
1954-01-04,NaN,NaN,NaN
1954-01-05,NaN,NaN,NaN
1954-01-06,NaN,NaN,NaN
1954-01-07,NaN,NaN,NaN
1954-01-08,NaN,NaN,NaN
...,...,...,...
2026-09-07,112.53,NaN,3.0
2026-09-08,112.44,NaN,3.0
2026-09-09,NaN,NaN,NaN


In [16]:
TICKERS = {
    "000001.SS": "SSE Composite Index",
}

OUT_DIR = Path("../2_data/2.1_raw/Yahoo Finance")
OUT_DIR.mkdir(parents=True, exist_ok=True)

for ticker, name in TICKERS.items():
    out_path = OUT_DIR / f"{ticker}_{name}_daily.csv"

    try:
        df = yf.download(
            ticker,
            period="max",
            interval="1d",
            auto_adjust=False,
            progress=False,
        )

        if df.empty:
            print(f"{ticker}: no data returned")
            continue

        df.index.name = "date"
        df.to_csv(out_path)

        print(
            f"{ticker}: {len(df):,} rows | "
            f"{df.index.min().date()} to {df.index.max().date()} | "
            f"Saved to {out_path.resolve()}"
        )

    except Exception as error:
        print(f"{ticker}: download failed — {error}")

000001.SS: 7,075 rows | 1997-07-02 to 2026-09-11 | Saved to C:\Users\Leo\PycharmProjects\Thesis Project\2_data\2.1_raw\Yahoo Finance\000001.SS_SSE Composite Index_daily.csv


In [19]:
df_target_features_china.index = pd.to_datetime(df_target_features_china.index)

adj_close = df["Adj Close"]
if isinstance(adj_close, pd.DataFrame):  
    adj_close = adj_close[ticker]

df_target_features_china["Local stock market index"] = (
    adj_close.reindex(df_target_features_china.index)
)

df_target_features_china

,FX rate,CPI growth,Local policy rate,Local stock market index
Date,,,,
1954-01-04,NaN,NaN,NaN,NaN
1954-01-05,NaN,NaN,NaN,NaN
1954-01-06,NaN,NaN,NaN,NaN
1954-01-07,NaN,NaN,NaN,NaN
1954-01-08,NaN,NaN,NaN,NaN
...,...,...,...,...
2026-09-07,112.53,NaN,3.0,3932.698975
2026-09-08,112.44,NaN,3.0,3940.551025
2026-09-09,NaN,NaN,NaN,3951.507080


In [18]:
output_path = Path("../2_data/2.2_processed/target_features_data/china.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)

df_target_features_china.to_csv(output_path, index=True)

print(f"Saved to: {output_path.resolve()}")

Saved to: C:\Users\Leo\PycharmProjects\Thesis Project\2_data\2.2_processed\target_features_data\china.csv
